## Physical Neural Informed Dehaing Model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from mamba_ssm import Mamba
import math

In [2]:
class PhysConvNeXtBlock(nn.Module):
    """
    ConvNeXt V2 Block: Best for local textures and edges.
    Includes Adaptive Layer Norm (AdaLN) for Time Embedding injection.
    """
    def __init__(self, dim, mult=2):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x, t_emb=None):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1) # (N, C, H, W) -> (N, H, W, C)
        
        # Adaptive Layer Norm (Time Injection)
        if t_emb is not None:
            # t_emb is (N, C) -> Scale & Shift
            x = self.norm(x)
            scale, shift = t_emb.chunk(2, dim=1)
            x = x * (1 + scale.unsqueeze(1).unsqueeze(1)) + shift.unsqueeze(1).unsqueeze(1)
        else:
            x = self.norm(x)

        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2) # (N, H, W, C) -> (N, C, H, W)
        return input + x

In [3]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.
    """
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        
        # We need Mamba backend

        # We use two separate Mamba mixers: one for forward, one for backward
        # Sharing weights is possible but separate usually performs better for vision
        self.mamba_fwd = Mamba(d_model=dim, d_state=64, d_conv=4, expand=2)
        self.mamba_bwd = Mamba(d_model=dim, d_state=64, d_conv=4, expand=2)

        # 2. THE FUSION LAYER (The upgrade)
        # Takes both directions (dim * 2) and learns how to combine them back to (dim)
        self.fusion_linear = nn.Linear(dim * 2, dim)
        
        # Optional: A Gate to let the network choose emphasis
        self.fusion_gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
    def forward(self, x, t_emb=None):
        """
        x: (B, C, H, W)
        """
        B, C, H, W = x.shape
        residual = x
        
        # 1. Prepare Sequence: (B, C, H, W) -> (B, L, C)
        x_flat = x.flatten(2).transpose(1, 2) # (B, L, C)
        x_norm = self.norm(x_flat)

        # 2. Inject Time
        if t_emb is not None:
             # Take only the first half (scale) for simple addition
            t_val, _ = t_emb.chunk(2, dim=1)
            # print(f'x_norm: {x_norm.shape}')
            # print(f't_val: {t_val.unsqueeze(1).shape}')
            x_norm = x_norm + t_val.unsqueeze(1)

        # 3. Bidirectional Scanning
        
        # --- Forward Scan (Standard) ---
        out_fwd = self.mamba_fwd(x_norm)
        
        # --- Backward Scan (Flip -> Scan -> Flip Back) ---
        x_flip = torch.flip(x_norm, dims=[1]) # Reverse the sequence
        out_bwd = self.mamba_bwd(x_flip)
        out_bwd = torch.flip(out_bwd, dims=[1]) # Reverse back to original order
        
        # 4. Combine
        # --- Learned Fusion (Better than Averaging) ---
        
        # Concatenate features: Shape becomes (B, L, 2*C)
        combined = torch.cat([out_fwd, out_bwd], dim=-1)
        
        # Calculate a Gate (0 to 1) deciding flow importance
        # "z" tells us how much to listen to the mixture
        z = self.fusion_gate(combined)
        
        # Project back to original dimension
        x_fused = self.fusion_linear(combined)
        
        # Gated Activation: This is very stable for Mamba
        x_out = x_fused * z
        
        # 5. Reshape back to Image
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        
        return residual + x_out

In [4]:
class FM_PhysMamba_UNET(nn.Module):
    def __init__(
        self, 
        in_channels=3, 
        base_dim=48, 
        dim_mults=[1, 2, 4, 8],
        physics_guided=True
    ):
        super().__init__()
        self.physics_guided = physics_guided
        self.dims = [base_dim * m for m in dim_mults]
        
        # --- Time Embedding ---
        time_dim = base_dim * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(base_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Time Projections (Adapting time vector to each layer size)
        self.down_time_projs = nn.ModuleList()
        self.up_time_projs = nn.ModuleList()

        # --- ENCODER ---
        self.init_conv = nn.Conv2d(in_channels, self.dims[0], 3, 1, 1)
        self.downs = nn.ModuleList()
        
        for i in range(len(self.dims)-1):
            dim_in, dim_out = self.dims[i], self.dims[i+1]
            
            self.down_time_projs.append(nn.Linear(time_dim, dim_in * 2))

            if i == 0:
                block = PhysConvNeXtBlock(dim_in)
            else:
                block = PhysBiMambaBlock(dim_in)
            
            # Encoder blocks: Pure ConvNeXt for efficiency
            self.downs.append(nn.ModuleList([
                PhysConvNeXtBlock(dim_in),
                block,
                nn.Conv2d(dim_in, dim_out, 4, 2, 1) # Downsample
            ]))

        # --- BOTTLENECK (The Brain) ---
        # This is where Mamba lives to understand global physics
        mid_dim = self.dims[-1]
        self.mid_time_proj = nn.Linear(time_dim, mid_dim * 2)
        
        self.mid_block1 = PhysBiMambaBlock(mid_dim) # Bidirectional Scan 1
        self.mid_block2 = PhysBiMambaBlock(mid_dim) # Bidirectional Scan 2
        
        # --- PHYSICS HEAD 1: ATMOSPHERE (A) ---
        # Predicts the global ambient light color (1x3 vector)
        self.atm_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(mid_dim, 64),
            nn.SiLU(),
            nn.Linear(64, 3), 
            nn.Sigmoid() 
        )

        # --- DECODER ---
        self.ups = nn.ModuleList()
        for i in range(len(self.dims)-2, -1, -1):
            dim_in, dim_out = self.dims[i+1], self.dims[i]
            
            self.up_time_projs.append(nn.Linear(time_dim, dim_out * 2))
            
            self.ups.append(nn.ModuleList([
                nn.ConvTranspose2d(dim_in, dim_out, 2, 2), # Upsample
                nn.Conv2d(dim_out*2, dim_out, 1), # Reduce concatenation
                # Hybrid: Mamba deep (for structure), ConvNeXt shallow (for texture)
                PhysBiMambaBlock(dim_out) if i > 0 else PhysConvNeXtBlock(dim_out),
                PhysConvNeXtBlock(dim_out)
            ]))

        # --- PHYSICS HEAD 2: TRANSMISSION (t) ---
        # Predicts the depth/haze map (HxW)
        self.trans_head = nn.Sequential(
            nn.Conv2d(self.dims[0], 16, 3, 1, 1),
            nn.SiLU(),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid() 
        )
        
        # --- MAIN OUTPUT: VELOCITY (v) ---
        self.final_conv = nn.Conv2d(self.dims[0], 3, 1)

    def get_sinusoidal_emb(self, t, device):
        # Creates standard sinusoidal position embeddings for time
        half_dim = self.dims[0] // 2
        
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def forward(self, x, t):
        # 1. Time Embedding
        t_emb_raw = self.get_sinusoidal_emb(t, x.device) 
        t_vec = self.time_mlp(t_emb_raw)
        
        # 2. Encoder Pass
        h = self.init_conv(x)
        skips = [h]
        
        for i, (block1, block2, down) in enumerate(self.downs):
            t_emb = self.down_time_projs[i](t_vec)
            h = block1(h, t_emb)
            h = block2(h, t_emb)
            skips.append(h)
            h = down(h)
            
        # 3. Bottleneck Pass (Global Physics)
        t_emb_mid = self.mid_time_proj(t_vec)
        h = self.mid_block1(h, t_emb_mid)
        h = self.mid_block2(h, t_emb_mid)
        
        # >>> EXTRACT PHYSICS: Atmosphere (A)
        # We pull this from the deepest features which contain the most global info
        A_pred = self.atm_head(h).view(-1, 3, 1, 1)
        
        # 4. Decoder Pass
        for i, (up, reduce, block1, block2) in enumerate(self.ups):
            h = up(h)
            skip = skips.pop()
            h = torch.cat([h, skip], dim=1) # Skip connection
            h = reduce(h)
            
            t_emb = self.up_time_projs[i](t_vec)
            h = block1(h, t_emb)
            h = block2(h, t_emb)
            
        # >>> EXTRACT PHYSICS: Transmission (t)
        # We pull this from the highest resolution features
        t_map = self.trans_head(h)
        
        # 5. Physics Guidance
        if self.physics_guided:
            # We explicitly multiply the features by (1 + t_map).
            # This boosts the signal in regions where transmission is high (clear),
            # and dampens it where transmission is low (hazy), guiding the restoration.
            h = h * (1 + t_map)

        # >>> EXTRACT MAIN: Velocity (v)
        v_pred = self.final_conv(h)
        
        return v_pred, t_map, A_pred

In [5]:
def get_phys_mamba_config(version="small"):
    if version == "small":
        # Fast training, lower memory. Good for debugging and RTX 3060/4060 class GPUs.
        return {
            "base_dim": 48,
            "dim_mults": [1, 2, 4, 8],  # 3 levels only (shallower)
            "physics_guided": True
        }
    
    elif version == "large":
        # SOTA performance. Heavy. Requires A100/A6000 or 24GB+ VRAM.
        return {
            "base_dim": 96,
            "dim_mults": [1, 2, 4, 8], # 4 levels (deeper)
            "physics_guided": True
        }
    
    else:
        raise ValueError(f"Unknown version: {version}")

### Profiling please

In [6]:
try:
    from thop import profile, clever_format
except ImportError:
    print("Please install thop: pip install thop")
    exit()

# def profile_version(version_name):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"\n--- Profiling: {version_name.upper()} ---")
    
#     # 1. Load Config
#     cfg = get_phys_mamba_config(version_name)
    
#     # 2. Instantiate Model
#     model = UltimatePhysMamba(**cfg).to(device)
#     model.eval()
    
#     # 3. Create Dummy Input (Standard Image Size: 256x256)
#     # Batch size 1 for profiling
#     input_tensor = torch.randn(1, 3, 256, 256).to(device)
#     time_tensor = torch.randn(1).to(device) # Random time step
    
#     # 4. Profile
#     # Note: We wrap the inputs in a tuple
#     macs, params = profile(model, inputs=(input_tensor, time_tensor), verbose=False)
    
#     # 5. Format and Print
#     macs_fmt, params_fmt = clever_format([macs, params], "%.3f")
#     print(f"Parameters: {params_fmt}")
#     print(f"MACs (GFLOPS): {macs_fmt}")
#     print(f"Base Dim: {cfg['base_dim']}")
#     print("-" * 30)

# if __name__ == "__main__":
#     profile_version("small")
#     profile_version("large")

def get_model_stats(version_name):
    print(f"--- Setting up {version_name.upper()} on GPU ---")
    
    # 1. Setup Device (Force GPU)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Mamba requires a GPU.")
    
    device = torch.device("cuda")
    
    # 2. Load Config & Model to GPU
    cfg = get_phys_mamba_config(version_name)
    model = UltimatePhysMamba(**cfg).to(device)
    model.eval()
    
    # 3. Dummy Data on GPU (B=1, C=3, H=256, W=256)
    x = torch.randn(1, 3, 256, 256).to(device)
    t = torch.randn(1).to(device)
    
    # 4. Compute MACs & Params (Running on GPU)
    # We pass the GPU model and GPU tensors directly to thop
    macs, params = profile(model, inputs=(x, t), verbose=False)
    
    # 5. Compute Memory (VRAM)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Inference Pass for Memory Measurement
    with torch.no_grad():
        _ = model(x, t)
        
    peak_bytes = torch.cuda.max_memory_allocated()
    mem_inf = f"{peak_bytes / (1024**2):.1f} MB"
    
    # Estimated Training Memory (Heuristic: ~3.5x Inference)
    mem_train = f"~{peak_bytes * 3.5 / (1024**2):.1f} MB"
        
    return cfg['base_dim'], params, macs, mem_inf, mem_train

def print_table():
    versions = ["small", "large"]
    
    # Header
    print(f"\n{'='*95}")
    print(f"{'VERSION':<10} | {'DIM':<5} | {'PARAMS':<15} | {'GFLOPS (MACs)':<15} | {'VRAM (Infer)':<15} | {'VRAM (Train)':<15}")
    print(f"{'-'*95}")
    
    for v in versions:
        try:
            base_dim, params, macs, mem_inf, mem_train = get_model_stats(v)
            
            # Format Numbers
            macs_fmt, params_fmt = clever_format([macs, params], "%.2f")
            
            # Print Row
            print(f"{v.upper():<10} | {base_dim:<5} | {params_fmt:<15} | {macs_fmt:<15} | {mem_inf:<15} | {mem_train:<15}")
        except Exception as e:
            print(f"{v.upper():<10} | FAILED: {str(e)}")
    
    print(f"{'='*95}\n")

if __name__ == "__main__":
    print_table()


VERSION    | DIM   | PARAMS          | GFLOPS (MACs)   | VRAM (Infer)    | VRAM (Train)   
-----------------------------------------------------------------------------------------------
--- Setting up SMALL on GPU ---
SMALL      | 48    | 7.29M           | 29.69G          | 194.5 MB        | ~680.9 MB      
--- Setting up LARGE on GPU ---
LARGE      | 96    | 27.55M          | 109.96G         | 441.4 MB        | ~1545.0 MB     



### Try testing some loss function

We recreate the Perceptual Loss function again but this time 
we only just \
the content loss (discards the style loss to reduce hallucination)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

In [8]:
class PerceptualLoss(nn.Module):
    """
    Computes Perceptual (Content) Loss using a frozen VGG16.
    
    CHANGES FROM ORIGINAL:
    1. Removed Style Loss (Gram Matrices) - unnecessary for Dehazing.
    2. Removed JSON dependency - hardcoded standard layers for stability.
    3. Added automatic input normalization.
    """
    def __init__(self, layer_indices = None):
        # Standard VGG16 Layers for Content Loss in Restoration:
        # relu1_2 (index 3), relu2_2 (index 8), relu3_3 (index 15), relu4_3 (index 22)
        if layer_indices is None:
            layer_indices = [3, 8, 15, 22]

        self.layer_indices = set(layer_indices)

        # Load VGG16 Backbone
        vgg = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1).features 
        vgg.eval()

        # Freeze parameters (we don't train VGG)
        for param in vgg.parameters():
            param.requires_grad = False 

        # Extract only the layers we need to save memory
        # We slice up to the max index we need
        max_idx = max(layer_indices)
        self.vgg_layers = vgg[:max_idx + 1]

        # ImageNet Normalizationo Constants 
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def normalize(self, x):
        """
        Normalize inputs to the range and stats VGG expects.
        Assumes input x is in range [0, 1] or [-1, 1].
        """
        # If the range of input is [-1, 1] convert to [0, 1]
        if x.min() < 0:
             x = (x + 1.0) / 2.0

        # Normalize with ImageNet mean/std
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        # 1 Normalize
        pred_norm = self.normalize(pred)
        target_norm = self.normalize(target)
        
        loss = 0.0
        x = pred_norm
        y = target_norm
        
        # 2. Pass through layers and accumulate loss
        for i, layer in enumerate(self.vgg_layers):
            x = layer(x)
            y = layer(y)
            
            if i in self.layer_indices:
                # Use L1 loss for features (sharper than MSE)
                loss += F.l1_loss(x, y)
                
        return loss

Testing the Charbornnier Loss

In [10]:
class CharbonnierLoss(nn.Module):
    """
    Robust L1 loss (softer near zero).
    Standard for image restoration tasks.
    """
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self).__init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.sqrt(diff * diff + self.eps**2)
        return torch.mean(loss)

# UltimateLoss

**The Mechanism:** This is a Threshold Penalty.

- 0.5 - A_pred: This calculates how far below 0.5 the prediction is.

- F.relu(...):

    - If A is 0.8 (Bright): 0.5 - 0.8 = -0.3. ReLU turns negatives to 0. No Loss. Good!

    - If A is 0.1 (Dark): 0.5 - 0.1 = 0.4. ReLU keeps positive 0.4. High Loss! Bad!

Why we need it: A represents the Atmospheric Light (the ambient light scattered by the haze). In almost all daytime hazy images, the haze is bright (white or light gray). Sometimes, a neural network might try to explain a dark region by saying "The atmosphere here is pitch black." This is physically impossible in a hazy day scene. This loss acts as a "guard rail," effectively screaming at the model: "The Atmosphere must be at least 0.5 brightness!"

In [11]:
class FM_PhysicalLoss(nn.Module):
    """
    Combines:
    1. Flow Matching Loss (Velocity)
    2. Physics Consistency Loss (Restoring Input I)
    3. Perceptual Loss (VGG Content)
    4. Smoothness Regularization
    """
    def __init__(self, loss_config = None):
        super().__init__()
        self.charbonnier = CharbonnierLoss()
        self.perceptual = PerceptualLoss()

        # --- Default Hyperparameters (Fallbacks) ---
        # If no config is provided, these defaults are used.
        self.weights = {
            "w_flow": 1.0,    # Velocity Matching
            "w_perc": 0.1,    # VGG Perceptual
            "w_phys": 0.2,    # Physics Consistency
            "w_tv": 0.01,     # Transmission Smoothness
            "w_atm": 0.01     # Atmosphere Constraint
        }

        # --- Override with Config ---
        if loss_config is not None:
            # We assume loss_config is a dict or an object like cfg.LOSS
            # We update our weights if the key exists in the config
            if hasattr(loss_config, "W_FLOW"): self.weights["w_flow"] = loss_config.W_FLOW
            if hasattr(loss_config, "W_PERC"): self.weights["w_perc"] = loss_config.W_PERC
            if hasattr(loss_config, "W_PHYS"): self.weights["w_phys"] = loss_config.W_PHYS
            if hasattr(loss_config, "W_TV"):   self.weights["w_tv"]   = loss_config.W_TV
            if hasattr(loss_config, "W_ATM"):  self.weights["w_atm"]  = loss_config.W_ATM
            
            # Support Dictionary access as well (if you pass a raw dict)
            if isinstance(loss_config, dict):
                self.weights.update(loss_config)
                
    def get_gradients(self, img):
        dy = img[:, :, 1:, :] - img[:, :, :-1, :]
        dx = img[:, :, :, 1:] - img[:, :, :, :-1]
        return dy, dx
    
    def forward(self, pred_tuple, target_clean, target_hazy):
        # Unpack predictions 
        v_pred, t_map, A_pred = pred_tuple;e 

        # --- A. Velocity Loss (Flow Matching Core) --- 
        # "Learn to move pixels from Hazy to Clean"
        target_v = target_clean - input_hazy 
        loss_v = self.charbonnier(v_pred, target_v) 

        # --- B. RECONSTRUCTED IMAGE -- 
        # J_pred = Hazy + Velocity 
        J_pred = input_hazy + v_pred

        # --- C. PERCEPTUAL LOSS (Visual Quality) ---
        # "Make the reconstructed image J look like a natural image"
        loss_percep = self.perceptual(J_pred, target_clean)

        # --- D. PHYSICS CONSISTENCY (Scientific Constraint) ---
        # "Do J, t, and A mathematically explain the input Hazy image?"
        # I = J*t + A*(1-t)
        I_reconstructed = J_pred * t_map  + A_pred * (1 - t_map)
        loss_phys = self.charbonnier(I_reconstructed, input_hazy)

        # --- E. REGULARIZERS ---
        # 1. TV Loss: Smoothness for Transmission map
        dy, dx = self.get_gradients(t_map)
        loss_tv = torch.mean(torch.abs(dy)) + torch.mean(torch.abs(dx))
        
        # 2. Atmosphere: Prevent pitch black A predictions 
        loss_atm = torch.mean(F.relu(0.5 - A_pred))

        # --- WEIGHTS ---
        # Adjust these if specific parts are failing
        total_loss = (self.weights["w_flow"] * loss_v) + \
                     (self.weights["w_perc"] * loss_percep) + \
                     (self.weights["w_phys"] * loss_phys) + \
                     (self.weights["w_tv"]   * loss_tv) + \
                     (self.weights["w_atm"]  * loss_atm)

        return total_loss, {
            "Total": total_loss.item(),
            "Flow": loss_v.item(),
            "VGG": loss_percep.item(),
            "Phys": loss_phys.item()
        }

## Testing everything

In [2]:
from model import FM_PhysMamba_UNET

In [ ]:
small_model = FM_PhysMamba_UNET(model_cfg_path="small")
large_model = FM_PhysMamba_UNET(model_cfg_path="large")